# Cell Type Annotation and Phenotyping

This notebook performs cell type identification and annotation using:
- Marker gene expression
- Automated annotation tools
- Manual curation
- scVI-tools for probabilistic modeling

**Input:** Preprocessed AnnData object from 01_preprocessing.ipynb

**Output:** Annotated AnnData with cell type labels

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Rapids GPU-accelerated single-cell ops (drop-in for sc.pp.scale/pca/neighbors,
# sc.tl.score_genes/leiden). UMAP intentionally stays on CPU per CLAUDE.md
# (rapids/cuML UMAP produced extreme outliers on this dataset previously).
import rapids_singlecell as rsc
import cupy as cp

# Enable TF32 / Tensor Cores on B200 (compute capability 10.0).
# Free ~30-50% speedup on FP32 matmul during scVI training.
if torch.cuda.is_available():
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)} (TF32 enabled, "
          f"matmul_precision={torch.get_float32_matmul_precision()})")

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white', frameon=False)

print(f"Scanpy version: {sc.__version__}")
print(f"scVI version: {scvi.__version__}")
print(f"rapids_singlecell version: {rsc.__version__}")


## 1. Load Preprocessed Data

In [ ]:
import os
from pathlib import Path
import numpy as np
import spatialdata as sd

SAMPLE_ID   = os.environ.get("XENIUM_SAMPLE_ID", "0041323")
DATA_DIR    = Path(os.environ.get("XENIUM_OUTPUT_DIR",  f"../data/processed/{SAMPLE_ID}"))
RAW_DIR     = Path(os.environ.get("XENIUM_RAW_DIR", "../data/raw"))
ZARR_PATH   = RAW_DIR / f"{SAMPLE_ID}.zarr"
OUTPUT_DIR  = DATA_DIR
FIGURES_DIR = Path(os.environ.get("XENIUM_FIGURES_DIR", f"../figures/{SAMPLE_ID}/02_phenotyping"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

prep = DATA_DIR / f"{SAMPLE_ID}_preprocessed.h5ad"

if prep.exists():
    print(f"Sample:    {SAMPLE_ID}")
    print(f"Loading preprocessed: {prep}")
    adata = sc.read_h5ad(prep)
    print(f"Loaded shape: {adata.shape}")
    if "leiden_1.5" in adata.obs.columns:
        print(f"Available clusters (leiden_1.5): {adata.obs['leiden_1.5'].nunique()}")
else:
    print(f"Sample:    {SAMPLE_ID}  (standalone — no preprocessed.h5ad)")
    print(f"Loading zarr: {ZARR_PATH}")
    if not ZARR_PATH.exists():
        raise FileNotFoundError(
            f"Neither {prep} nor {ZARR_PATH} exists. Run 00_ingest_xenium.ipynb first."
        )
    sdata = sd.read_zarr(ZARR_PATH)
    adata = sdata.tables["table"]

    # Drop Xenium control probes BEFORE QC
    control_mask = adata.var_names.str.match(
        r"^(NegControlProbe_|UnassignedCodeword_|NegControlCodeword_|antisense_|BLANK_)",
        case=False,
    )
    if control_mask.any():
        print(f"  dropping {int(control_mask.sum())} control probes")
        adata = adata[:, ~control_mask].copy()

    # QC metrics + filter (10x 5k tutorial alignment, same thresholds as 01).
    # Upper bound is DENSITY-based (counts / cell_area > q98) not absolute counts —
    # in FFPE Xenium, large healthy cells (acinar, ductal) have high counts and an
    # absolute cap discards them. See 01_preprocessing_v2.ipynb cell 14 comment.
    sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)
    print(f"  before filter: {adata.shape}")
    sc.pp.filter_cells(adata, min_genes=20)
    sc.pp.filter_cells(adata, min_counts=50)
    density = (
        adata.obs["total_counts"].astype(np.float64)
        / np.maximum(adata.obs["cell_area"].astype(np.float64), 1.0)
    )
    density_cutoff = float(np.quantile(density, 0.98))
    print(f"  density cutoff (q98 of counts/area) = {density_cutoff:.3f}")
    adata = adata[(density <= density_cutoff).to_numpy()].copy()
    sc.pp.filter_genes(adata, min_cells=100)
    print(f"  after filter:  {adata.shape}")

    # Preserve raw counts BEFORE any normalization. scVI in cell 17 reads layer='counts'.
    adata.layers["counts"] = adata.X.copy()
    print(f"  counts layer stored")
    print(f"Loaded shape: {adata.shape}")


## 2. Find Cluster Markers

In [ ]:
# Always ensure lognorm + log-normalized .X for downstream score_genes.
# rank_genes_groups (marker discovery by leiden_1.5) only runs if 01_preprocessing
# already produced leiden_1.5 clusters. Without that, scoring proceeds via the
# panel-aware marker dict and we skip the per-cluster DE diagnostic.
if "lognorm" not in adata.layers:
    adata.X = adata.layers["counts"].copy()
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    adata.layers["lognorm"] = adata.X.copy()
else:
    adata.X = adata.layers["lognorm"].copy()

if "leiden_1.5" in adata.obs.columns:
    print("leiden_1.5 present — running rank_genes_groups for marker discovery")
    n_per_group = 1000
    np.random.seed(42)
    indices = []
    for group in adata.obs["leiden_1.5"].unique():
        group_idx = adata.obs.index[adata.obs["leiden_1.5"] == group]
        indices.extend(np.random.choice(group_idx, size=min(n_per_group, len(group_idx)), replace=False))
    adata_de = adata[indices].copy()
    sc.tl.rank_genes_groups(
        adata_de,
        groupby="leiden_1.5",
        method="wilcoxon",
        layer="lognorm",
        key_added="rank_genes_leiden",
        use_raw=False,
    )
    adata.uns["rank_genes_leiden"] = adata_de.uns["rank_genes_leiden"]
    sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False, key="rank_genes_leiden")
    plt.savefig(FIGURES_DIR / "marker_genes_by_cluster.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("Marker gene analysis completed")
else:
    print("[skipped] leiden_1.5 not in adata.obs — skipping rank_genes diagnostic.")
    print("          scVI in cell 17 will compute leiden_scvi for downstream consensus.")


In [ ]:
if "rank_genes_leiden" in adata.uns:
    result = adata.uns["rank_genes_leiden"]
    groups = result["names"].dtype.names
    marker_df = pd.DataFrame({
        group + "_" + key: result[key][group]
        for group in groups for key in ["names", "scores"]
    })
    marker_df.to_csv(OUTPUT_DIR / f"{SAMPLE_ID}_marker_genes.csv")
    print(f"Marker genes saved to {OUTPUT_DIR / f'{SAMPLE_ID}_marker_genes.csv'}")
    print("\nTop 5 marker genes per cluster:")
    for cluster in groups[:5]:
        print(f"\nCluster {cluster}:")
        print(marker_df[f"{cluster}_names"].head().tolist())
else:
    print("[skipped] no rank_genes_leiden — see cell 5.")


## 3. Define Cell Type Markers

Define known marker genes for major cell types (customize based on tissue type)

In [6]:
adata

AnnData object with n_obs × n_vars = 521984 × 5081
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_genes', 'n_counts', 'leiden_1.5'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'mean', 'std', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'dendrogram_leiden_1.5', 'hvg', 'leiden_1.5', 'leiden_1.5_centrality_scores', 'leiden_1.5_colors', 'leiden_1.5_nhood_enrichment', 'log1p',

In [ ]:
if "rank_genes_leiden" in adata.uns:
    def _add_group_separators(dp_or_mp, group_dict_or_n_per_group, n_groups=None,
                              color="black"):
        """Draw vertical lines BETWEEN gene groups on a scanpy DotPlot/MatrixPlot.
        Pass the object returned by sc.pl.* with return_fig=True. We force-render
        via make_figure() if needed, then add axvlines to the mainplot_ax at the
        correct boundary positions (scanpy places genes at 0.5, 1.5, 2.5, ...
        so boundaries between groups are at integer cumulative gene counts).
        """
        if dp_or_mp.ax_dict is None:
            dp_or_mp.make_figure()
        main_ax = dp_or_mp.ax_dict.get("mainplot_ax")
        if main_ax is None:
            return
        if isinstance(group_dict_or_n_per_group, dict):
            cum = 0
            items = list(group_dict_or_n_per_group.items())
            for i, (_, genes) in enumerate(items):
                cum += len(genes)
                if i < len(items) - 1:
                    main_ax.axvline(cum, color=color, linewidth=1.6, alpha=0.95, zorder=10)
        else:
            n_per = int(group_dict_or_n_per_group)
            for k in range(1, int(n_groups)):
                main_ax.axvline(k * n_per, color=color, linewidth=1.6, alpha=0.95, zorder=10)
    hm = sc.pl.rank_genes_groups_heatmap(
        adata, n_genes=5, key="rank_genes_leiden", show_gene_labels=True,
        figsize=(28, 28), cmap="viridis", vmin=-1, vmax=3,
        layer="lognorm", swap_axes=True, var_group_rotation=90, show=False,
        return_fig=True,
    )
    n_clusters = adata.obs["leiden_1.5"].astype("category").cat.categories.size if "leiden_1.5" in adata.obs.columns else 0
    if n_clusters >= 2 and hasattr(hm, "ax_dict"):
        _add_group_separators(hm, 5, n_groups=n_clusters, color="white")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "marker_genes_heatmap.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("[skipped] marker_genes_heatmap — no rank_genes_leiden.")


In [ ]:
# Panel-aware pancreas markers for the T1D-focused hAtlas_v1.1+100 panel.
# Absent from panel: INS, GCG, SST, PPY, TTR, IRX2, PRSS1/2, CPA1/2, CTRB1/2,
# CELA1/2A/3A, REG1A/3A. Beta uses surrogates; Alpha/Delta are flagged as low
# confidence; PP is dropped; Acinar reduces to AMY1A; Endocrine is a pan-islet
# fallback used when hormone-subtype resolution fails.
pancreas_markers = {
    'Beta':          ['IAPP', 'MAFA', 'NKX6-1', 'PDX1', 'SLC30A8', 'ABCC8', 'KCNJ11',
                      'GLP1R', 'PCSK1'],
    'Alpha':         ['ARX', 'MAFB'],
    'Delta':         ['HHEX', 'LEPR'],
    'Epsilon':       ['GHRL'],
    'Endocrine':     ['CHGA', 'INSM1', 'ISL1', 'NEUROD1', 'FEV', 'PAX6', 'SCG5'],
    'Acinar':        ['AMY1A'],
    'Ductal':        ['KRT19', 'SOX9', 'HNF1B', 'MUC1', 'CFTR'],
    'Stellate':      ['ACTA2', 'PDGFRB', 'RGS5', 'PDGFRA'],
    'Endothelial':   ['PECAM1', 'CDH5', 'CLDN5', 'PLVAP', 'KDR', 'ENG'],
    'T_cells':       ['CD3D', 'CD3E', 'CD3G', 'CD8A', 'CD8B', 'CD4', 'PTPRC', 'FOXP3'],
    'B_cells':       ['MS4A1', 'CD79A', 'CD79B', 'CD19'],
    'Myeloid':       ['CD68', 'CD163', 'CSF1R', 'MARCO', 'CD14'],
    'Schwann':       ['MPZ', 'SOX10', 'NES'],
    'Proliferating': ['MKI67', 'TOP2A', 'PCNA'],
}

# Low-confidence types where panel coverage is thin — used to downgrade ambiguous assignments.
LOW_CONFIDENCE_TYPES = {'Alpha', 'Delta', 'Epsilon', 'Acinar'}

for ct, genes in pancreas_markers.items():
    missing = [g for g in genes if g not in adata.var_names]
    if missing:
        print(f'{ct}: missing {missing}')
available_markers = {
    ct: [g for g in genes if g in adata.var_names]
    for ct, genes in pancreas_markers.items()
    if any(g in adata.var_names for g in genes)
}
print(f"\nCell types with at least one probe in panel: {list(available_markers.keys())}")


In [ ]:
if "leiden_1.5" in adata.obs.columns:
    def _add_group_separators(dp_or_mp, group_dict_or_n_per_group, n_groups=None,
                              color="black"):
        """Draw vertical lines BETWEEN gene groups on a scanpy DotPlot/MatrixPlot.
        Pass the object returned by sc.pl.* with return_fig=True. We force-render
        via make_figure() if needed, then add axvlines to the mainplot_ax at the
        correct boundary positions (scanpy places genes at 0.5, 1.5, 2.5, ...
        so boundaries between groups are at integer cumulative gene counts).
        """
        if dp_or_mp.ax_dict is None:
            dp_or_mp.make_figure()
        main_ax = dp_or_mp.ax_dict.get("mainplot_ax")
        if main_ax is None:
            return
        if isinstance(group_dict_or_n_per_group, dict):
            cum = 0
            items = list(group_dict_or_n_per_group.items())
            for i, (_, genes) in enumerate(items):
                cum += len(genes)
                if i < len(items) - 1:
                    main_ax.axvline(cum, color=color, linewidth=1.6, alpha=0.95, zorder=10)
        else:
            n_per = int(group_dict_or_n_per_group)
            for k in range(1, int(n_groups)):
                main_ax.axvline(k * n_per, color=color, linewidth=1.6, alpha=0.95, zorder=10)
    dp = sc.pl.dotplot(
        adata, available_markers, groupby="leiden_1.5",
        dendrogram=True, cmap="Reds", figsize=(18, 10),
        vmin=0.0, vmax=1.0, standard_scale="var",
        swap_axes=False, var_group_rotation=90, show=False,
        return_fig=True,
    )
    _add_group_separators(dp, available_markers, color="black")
    plt.savefig(FIGURES_DIR / "dotplot_markers_by_cluster.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("[skipped] leiden_1.5 dotplot — will plot by leiden_scvi in cell 18 after scVI runs.")


## 5. Score Cell Types

In [ ]:
# Score cells for each cell type using the log-normalized layer.
# Plot deferred to AFTER cell 17 (where scVI computes X_umap). Plotting here
# would render scores on the stale Xenium-builtin UMAP from sd.read_zarr.
for cell_type, genes in available_markers.items():
    score_name = f"{cell_type}_score"
    sc.tl.score_genes(adata, genes, score_name=score_name, use_raw=False)
    print(f"Scored {cell_type}")

print("Score columns ready; UMAP plot deferred until cell 17 finishes scVI UMAP.")


## 6. Assign Cell Types Based on Scores

In [ ]:
# Assign cell types by highest score; flag low-confidence assignments for types
# whose panel coverage is thin (Alpha/Delta/Epsilon/Acinar). Threshold is
# per-dataset (median - 1*MAD on celltype_confidence).
#
# After initial argmax assignment, run an "anti-acinar override" pass:
# the panel has ONLY AMY1A as the acinar marker, so any cell with detectable
# AMY1A can argmax-out as Acinar even when it's actually expressing strong,
# specific markers of another lineage (Ductal KRT19/CFTR, Endothelial
# PECAM1/CDH5/CLDN5, Stellate ACTA2/PDGFRB, immune CD3D/CD68, etc.).
# For each Acinar-called cell, we check expression of curated SPECIFIC alt
# markers in the lognorm layer; if any alt set's mean exceeds the median
# across cells already confidently labeled that type, we reassign.

from scipy.sparse import issparse as _issparse

# Curated SPECIFIC anti-acinar marker sets (must NOT include genes that bleed
# across cell types). Each set is a small high-confidence panel.
ANTI_ACINAR_MARKERS = {
    "Beta":        ["IAPP", "MAFA", "NKX6-1", "PDX1", "SLC30A8"],
    "Endocrine":   ["CHGA", "INSM1", "ISL1", "NEUROD1", "FEV"],
    "Ductal":      ["KRT19", "CFTR", "MUC1"],
    "Stellate":    ["ACTA2", "PDGFRB", "RGS5"],
    "Endothelial": ["PECAM1", "CDH5", "CLDN5", "PLVAP"],
    "T_cells":     ["CD3D", "CD3E", "CD8A", "PTPRC"],
    "B_cells":     ["MS4A1", "CD79A"],
    "Myeloid":     ["CD68", "CD163", "MARCO"],
    "Schwann":     ["MPZ", "SOX10"],
}

def _per_cell_marker_mean(adata_, gene_list, layer="lognorm"):
    """Per-cell mean of `gene_list` from layers[layer]. Returns 1D np array."""
    present = [g for g in gene_list if g in adata_.var_names]
    if not present:
        return None
    X = adata_[:, present].layers[layer]
    if _issparse(X):
        return np.asarray(X.mean(axis=1)).ravel()
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = [f"{ct}_score" for ct in available_markers.keys()]

if score_cols:
    scores_df = adata.obs[score_cols]
    adata.obs["predicted_celltype"] = scores_df.idxmax(axis=1).str.replace("_score", "")
    top2_scores = np.sort(scores_df.values, axis=1)[:, -2:]
    adata.obs["celltype_confidence"] = top2_scores[:, 1] - top2_scores[:, 0]

    # ---------- Anti-acinar override pass ----------
    print("\n--- Anti-acinar override ---")
    pred = adata.obs["predicted_celltype"].astype(str).to_numpy()
    n_acinar_initial = int((pred == "Acinar").sum())
    print(f"  initial Acinar calls: {n_acinar_initial:,}")

    # Pre-compute alt marker means for each candidate alt type AND a baseline
    # threshold (median of that mean across cells confidently called the alt type).
    alt_mean = {}
    alt_thr = {}
    for alt_ct, marker_list in ANTI_ACINAR_MARKERS.items():
        m = _per_cell_marker_mean(adata, marker_list, layer="lognorm")
        if m is None or m.size == 0:
            continue
        alt_mean[alt_ct] = m
        # Threshold: median of `m` over cells initially called `alt_ct`.
        # Fallback: 75th percentile of `m` across all cells (so we don't override on noise).
        ct_initial_mask = (pred == alt_ct)
        if ct_initial_mask.sum() >= 50:
            alt_thr[alt_ct] = float(np.median(m[ct_initial_mask]))
        else:
            alt_thr[alt_ct] = float(np.quantile(m, 0.75))
        print(f"  {alt_ct}: marker_mean threshold = {alt_thr[alt_ct]:.3f} "
              f"(initial cells of this type: {int(ct_initial_mask.sum()):,})")

    # For each Acinar-called cell, find the alt type with the largest
    # (alt_mean - alt_thr) margin (only positives count). Reassign if any positive.
    acinar_mask = (pred == "Acinar")
    if acinar_mask.any() and alt_mean:
        # Stack alt margins as a (n_alt_types, n_cells) array; only consider Acinar cells
        alt_names = list(alt_mean.keys())
        margins = np.stack([alt_mean[a] - alt_thr[a] for a in alt_names], axis=0)  # (T, N)
        # For non-Acinar cells, mask margins to -inf so they aren't affected
        margins[:, ~acinar_mask] = -np.inf
        best_alt_idx = margins.argmax(axis=0)
        best_alt_margin = margins[best_alt_idx, np.arange(len(pred))]
        # Reassign Acinar cells whose best alt margin > 0
        reassign = acinar_mask & (best_alt_margin > 0)
        n_reassigned = int(reassign.sum())
        print(f"  Acinar cells with positive alt margin: {n_reassigned:,} ({100*n_reassigned/max(n_acinar_initial,1):.1f}%)")
        if n_reassigned > 0:
            new_labels = np.array(alt_names)[best_alt_idx]
            pred[reassign] = new_labels[reassign]
            # Per-type breakdown
            unique, counts = np.unique(new_labels[reassign], return_counts=True)
            for u, c in sorted(zip(unique, counts), key=lambda kv: -kv[1]):
                print(f"    -> {u}: +{int(c):,}")
        adata.obs["predicted_celltype"] = pd.Categorical(pred)

    # ---------- Indeterminate flagging (post-override) ----------
    print("\n--- Confidence-based Indeterminate flagging ---")
    conf = adata.obs["celltype_confidence"].to_numpy()
    pred = adata.obs["predicted_celltype"].astype(str).to_numpy()
    low_conf = pred.copy()
    for ct in LOW_CONFIDENCE_TYPES:
        mask = pred == ct
        if mask.sum() < 10:
            continue
        c = conf[mask]
        med = np.median(c)
        mad = np.median(np.abs(c - med)) + 1e-12
        thr = med - mad
        is_indet = mask & (conf < thr)
        bucket = "Indeterminate_endocrine" if ct in {"Alpha", "Delta", "Epsilon"} else "Indeterminate_exocrine"
        low_conf[is_indet] = bucket
        print(f"  {ct}: flagged {int(is_indet.sum())}/{int(mask.sum())} as {bucket} (thr conf<{thr:.3f})")
    adata.obs["predicted_celltype"] = pd.Categorical(low_conf)

    print("\nCell type distribution (post-anti-acinar + confidence-gate):")
    print(adata.obs["predicted_celltype"].value_counts())

    # Plot deferred to cell 17 (so it lands on the scVI UMAP, not the stale Xenium UMAP).


## 7. scVI-tools for Advanced Analysis (Optional)

Use scVI for probabilistic modeling and better integration

In [ ]:
# scVI on B200 GPU (TF32 enabled in cell 1) -> rapids_singlecell neighbors
# (cuML kNN, GPU) -> scanpy CPU UMAP -> scanpy CPU leiden (igraph flavor).
# Library-size effect is modelled internally by scVI's NB likelihood; passing
# total_counts as a covariate here would double-regress it. cell_area (FFPE
# segmentation size) IS a valid per-cell nuisance covariate to hand scVI.
# UMAP intentionally stays on CPU: rapids/cuML UMAP produced extreme outliers
# on this dataset previously (see CLAUDE.md). Leiden on CPU because cugraph
# leiden currently has a dask-cuda<->dask version drift.
import torch

scvi.model.SCVI.setup_anndata(
    adata,
    layer='counts',
    batch_key=None,
    continuous_covariate_keys=['cell_area'],
)
vae = scvi.model.SCVI(adata, n_layers=2, n_latent=30, gene_likelihood='nb', dropout_rate=0.1)

print("Training scVI model ...")
use_gpu = torch.cuda.is_available()
print(f"  CUDA available: {use_gpu}")
vae.train(
    max_epochs=200,
    early_stopping=True,
    early_stopping_patience=20,
    accelerator=('gpu' if use_gpu else 'cpu'),
    devices=1,
    batch_size=4096,
    datasplitter_kwargs={"num_workers": 8, "pin_memory": True, "persistent_workers": True},
)
adata.obsm['X_scvi'] = vae.get_latent_representation()

# Primary embedding: scVI latent -> cosine neighbours -> UMAP -> leiden (igraph flavor).
# Parameters aligned with 10x Xenium 5k tutorial (2026).
print("scVI neighbors/UMAP/leiden (scanpy, CPU, cosine, min_dist=0.5, igraph) ...")
rsc.pp.neighbors(adata, use_rep='X_scvi', n_neighbors=30, metric='cosine', random_state=0, key_added='scvi')  # GPU
sc.tl.umap(adata, neighbors_key='scvi', min_dist=0.5, spread=1.0, random_state=0)
sc.tl.leiden(
    adata, resolution=1.0, neighbors_key='scvi', key_added='leiden_scvi',
    flavor='igraph', n_iterations=-1, directed=False, random_state=0,
)

# ---- Validation overlay: library size MUST NOT drive the embedding ----
um = adata.obsm['X_umap']
from scipy.stats import spearmanr as _spearmanr
_r1, _ = _spearmanr(um[:, 0], adata.obs['total_counts'])
_r2, _ = _spearmanr(um[:, 1], adata.obs['total_counts'])
print(f"  Spearman(UMAP axis vs total_counts): {_r1:.3f} (x), {_r2:.3f} (y)")
if max(abs(_r1), abs(_r2)) > 0.3:
    print(f"  WARNING: library size still leaks into scVI UMAP (|rho|={max(abs(_r1),abs(_r2)):.2f}>0.3).")
    print(f"  The PCA-fallback UMAP will be checked as a sanity backup.")

# Color scaling: clip continuous metrics at the 95th percentile so the bulk
# distribution spans the full viridis range (otherwise a few extreme cells
# squash 95%+ of cells into the dark-purple end and the plot looks uniform).
import numpy as _np
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for col, ax, title in [
    ("total_counts",      axes[0, 0], "Total counts"),
    ("n_genes_by_counts", axes[0, 1], "N genes/cell"),
    ("cell_area",         axes[1, 0], "Cell area"),
]:
    vmax = float(_np.quantile(adata.obs[col], 0.95))
    sc.pl.umap(adata, color=col, ax=ax, show=False, cmap="viridis",
               title=f"{title} (clipped p95={vmax:.0f})",
               vmin=float(adata.obs[col].min()), vmax=vmax)
sc.pl.umap(adata, color='leiden_scvi', ax=axes[1, 1], show=False,
           title='leiden_scvi', legend_loc='on data', legend_fontsize=11,
           legend_fontoutline=2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'umap_qc_overlay.png', dpi=200, bbox_inches='tight')
plt.show()

# ---- Independent PCA-fallback embedding (10x-canonical recipe) ----
# Stored as X_umap_pca and leiden_pca; gives us a non-scVI reference in case scVI fails.
if 'X_pca' in adata.obsm:
    print("Fallback: PCA->UMAP (10x 5k recipe) ...")
    adata.obsm['X_umap_scvi'] = adata.obsm['X_umap'].copy()
    rsc.pp.neighbors(adata, n_neighbors=15, metric='cosine', use_rep='X_pca', random_state=0, key_added='pca')  # GPU
    sc.tl.umap(adata, neighbors_key='pca', min_dist=0.5, spread=1.0, random_state=0)
    adata.obsm['X_umap_pca'] = adata.obsm['X_umap'].copy()
    sc.tl.leiden(
        adata, resolution=1.0, neighbors_key='pca', key_added='leiden_pca',
        flavor='igraph', n_iterations=-1, directed=False, random_state=0,
    )
    # Restore scVI UMAP as primary .obsm['X_umap']
    adata.obsm['X_umap'] = adata.obsm['X_umap_scvi']
    print("  X_umap_scvi, X_umap_pca, leiden_scvi, leiden_pca all stored.")
else:
    print("Skipping PCA fallback: X_pca not present in preprocessed h5ad.")

print("scVI phenotyping embedding completed.")


# ---- Render celltype score UMAPs using the scVI UMAP (deferred from cell 13) ----
score_cols = [c for c in adata.obs.columns if c.endswith("_score")]
if score_cols:
    n = len(score_cols); ncols = 3; nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).flatten()
    for i, s in enumerate(score_cols):
        sc.pl.umap(adata, color=s, ax=axes[i], show=False, cmap="viridis", title=s, size=2, frameon=False)
    for j in range(n, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "celltype_scores_umap.png", dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  celltype_scores_umap.png rendered on scVI UMAP ({len(score_cols)} score cols)")


# ---- Render predicted_celltypes.png on the scVI UMAP (deferred from cell 15) ----
if "predicted_celltype" in adata.obs.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sc.pl.umap(adata, color="predicted_celltype", ax=axes[0], show=False, size=2, frameon=False)
    if "celltype_confidence" in adata.obs.columns:
        sc.pl.umap(adata, color="celltype_confidence", ax=axes[1], show=False, cmap="viridis", size=2, frameon=False)
    else:
        axes[1].axis("off")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "predicted_celltypes.png", dpi=200, bbox_inches="tight")
    plt.close()
    print("  predicted_celltypes.png rendered on scVI UMAP")


In [ ]:
# Compare scVI clustering with marker-based prediction
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sc.pl.umap(adata, color='leiden_scvi', ax=axes[0], show=False, title='scVI Leiden')
sc.pl.umap(adata, color='predicted_celltype', ax=axes[1], show=False, title='Marker-based prediction')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'scvi_vs_markers.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Manual Refinement (if needed)

In [ ]:
# Auto-derive per-cluster consensus cell type from the per-cell `predicted_celltype` (score-based).
# Take the majority label within each leiden_scvi cluster; this gives clean, discretized labels
# without requiring manual inspection of the marker heatmap.
# Override the auto map by populating `manual_override` for individual clusters after reviewing
# figures/.../02_phenotyping/marker_genes_heatmap.png.

auto_annotation = (
    adata.obs.groupby('leiden_scvi', observed=True)['predicted_celltype']
    .agg(lambda s: s.value_counts().idxmax())
    .to_dict()
)

# Per-cluster purity (fraction of cells matching the consensus label) — low purity flags mixed clusters
purity = (
    adata.obs.groupby('leiden_scvi', observed=True)['predicted_celltype']
    .agg(lambda s: s.value_counts(normalize=True).iloc[0])
    .to_dict()
)
cluster_sizes = adata.obs['leiden_scvi'].value_counts().to_dict()

print(f"{'cluster':>8}  {'size':>8}  {'consensus':>14}  purity")
for cl in sorted(auto_annotation, key=lambda k: int(k) if str(k).isdigit() else k):
    print(f"{cl:>8}  {cluster_sizes[cl]:>8}  {auto_annotation[cl]:>14}  {purity[cl]:.2f}")

# Populate this dict to override auto assignments for specific clusters after inspecting markers.
# Keys: leiden_scvi cluster id (str); values: pancreas cell type.
manual_override = {
    # '0': 'Acinar',
    # '7': 'Beta',
}

cluster_annotation = {**auto_annotation, **manual_override}

adata.obs['manual_celltype'] = adata.obs['leiden_scvi'].map(cluster_annotation).astype('category')
print(f"\nAssigned {adata.obs['manual_celltype'].nunique()} cell types across "
      f"{adata.obs['leiden_scvi'].nunique()} clusters")

sc.pl.umap(adata, color='manual_celltype', show=False)
plt.savefig(FIGURES_DIR / 'cluster_consensus_annotation.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Final Cell Type Assignment

In [ ]:
adata.obs['celltype'] = adata.obs['manual_celltype']

print("\nFinal cell type distribution:")
print(adata.obs['celltype'].value_counts())

# Color scaling: clip total_counts and n_genes at p95 so contrast is visible
# across the bulk distribution. Larger legend fonts so categorical labels are
# readable on 1.2M-cell UMAPs.
import numpy as _np
fig, axes = plt.subplots(2, 2, figsize=(15, 15))
sc.pl.umap(adata, color='celltype', ax=axes[0, 0], show=False,
           title='Cell Types', legend_loc='on data', legend_fontsize=11,
           legend_fontoutline=2)
sc.pl.umap(adata, color='leiden_scvi', ax=axes[0, 1], show=False,
           title='scVI Leiden', legend_loc='on data', legend_fontsize=11,
           legend_fontoutline=2)
for col, ax, title in [
    ("total_counts",      axes[1, 0], "Total Counts"),
    ("n_genes_by_counts", axes[1, 1], "N Genes"),
]:
    vmax = float(_np.quantile(adata.obs[col], 0.95))
    sc.pl.umap(adata, color=col, ax=ax, show=False, cmap="viridis",
               title=f"{title} (clipped p95={vmax:.0f})",
               vmin=float(adata.obs[col].min()), vmax=vmax)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_celltypes_overview.png', dpi=300, bbox_inches='tight')
plt.show()


# ---- Lineage phenotyping: refine multi-lineage cells via canary-marker depth ----
# For each cell, count how many DISTINCT lineage-specific markers each canary
# panel detects (raw count >= 3). Cells with multiple exclusive groups detected
# are resolved to their dominant lineage (top-depth) when the margin >= 2 and
# top_depth >= 2. Cells flagged this way are tagged "lineage_phenotyped".
# Cells with >= 3 mutually-exclusive groups are tagged "doublet_suspected"
# (kept as-is — no resolution attempted).
# See scripts/verify_upper_tail_doublets.py and scripts/lineage_phenotype.py
# for the standalone post-hoc form.

LINEAGE_CANARY_PANELS = {
    "Acinar":        ["AMY1A", "CUZD1"],
    "Ductal":        ["KRT19", "CFTR", "MUC1"],
    "Beta":          ["IAPP", "MAFA", "NKX6-1", "PDX1", "SLC30A8"],
    "Alpha":         ["ARX", "MAFB"],
    "Endocrine_pan": ["CHGA", "INSM1", "ISL1", "NEUROD1", "FEV"],
    "Stellate":      ["ACTA2", "PDGFRB", "RGS5"],
    "Endothelial":   ["PECAM1", "CDH5", "CLDN5", "PLVAP"],
    "T_cells":       ["CD3D", "CD3E", "CD8A", "PTPRC"],
    "B_cells":       ["MS4A1", "CD79A"],
    "Myeloid":       ["CD68", "CD163", "MARCO"],
    "Schwann":       ["MPZ", "SOX10"],
}
LINEAGE_EXCLUSIVE_GROUP = {
    "Acinar": "Exocrine_epi", "Ductal": "Exocrine_epi",
    "Beta": "Endocrine_islet", "Alpha": "Endocrine_islet",
    "Endocrine_pan": "Endocrine_islet",
    "Stellate": "Stromal", "Endothelial": "Endothelial",
    "T_cells": "Lymphoid", "B_cells": "Lymphoid",
    "Myeloid": "Myeloid", "Schwann": "Neural",
}
# Detection threshold on the LOGNORM layer (sample-comparable across
# samples with different sequencing depth). log1p(3) ≈ 1.386.
LOG_DETECT = float(np.log1p(3.0))
DOMINANCE_MARGIN = 2

if "counts" in adata.layers:
    _gene_idx = {g: i for i, g in enumerate(adata.var_names)}
    _all_canary = sorted({g for genes in LINEAGE_CANARY_PANELS.values() for g in genes})
    _present = [g for g in _all_canary if g in _gene_idx]
    _cols = [_gene_idx[g] for g in _present]
    _pos = {g: j for j, g in enumerate(_present)}
    # Use lognorm layer for sample-comparable thresholding (computed earlier in cell 5).
    _layer = adata.layers["lognorm"] if "lognorm" in adata.layers else adata.layers["counts"]
    _canary = _layer[:, _cols].toarray() if hasattr(_layer, "toarray") else np.asarray(_layer[:, _cols])
    _canary = np.asarray(_canary)

    _lineages = list(LINEAGE_CANARY_PANELS.keys())
    _depth = np.zeros((adata.n_obs, len(_lineages)), dtype=np.int8)
    _sum = np.zeros((adata.n_obs, len(_lineages)), dtype=np.float32)
    for _li, _lin in enumerate(_lineages):
        _idxs = [_pos[g] for g in LINEAGE_CANARY_PANELS[_lin] if g in _pos]
        if not _idxs:
            continue
        _sub = _canary[:, _idxs]
        _depth[:, _li] = (_sub >= LOG_DETECT).sum(axis=1).astype(np.int8)
        _sum[:, _li] = _sub.sum(axis=1)

    _detected = _depth >= 1
    _group_names = sorted(set(LINEAGE_EXCLUSIVE_GROUP[lin] for lin in _lineages))
    _group_to_idx = {g: i for i, g in enumerate(_group_names)}
    _lin_group = np.array([_group_to_idx[LINEAGE_EXCLUSIVE_GROUP[lin]] for lin in _lineages])
    _group_hit = np.zeros((adata.n_obs, len(_group_names)), dtype=bool)
    for _li in range(len(_lineages)):
        if not _detected[:, _li].any():
            continue
        _group_hit[:, _lin_group[_li]] |= _detected[:, _li]
    _n_groups = _group_hit.sum(axis=1).astype(np.int8)

    _composite = _depth.astype(np.float64) * 1000.0 + _sum.astype(np.float64)
    _top = _composite.argmax(axis=1)
    _td = _depth[np.arange(adata.n_obs), _top]
    _composite2 = _composite.copy()
    _composite2[np.arange(adata.n_obs), _top] = -1
    _second = _composite2.argmax(axis=1)
    _sd = _depth[np.arange(adata.n_obs), _second]
    _margin = _td.astype(np.int16) - _sd.astype(np.int16)

    _status = np.empty(adata.n_obs, dtype=object)
    _is_doublet = _n_groups >= 3
    _is_multi = (_n_groups == 2) & ~_is_doublet
    _is_single = _n_groups <= 1
    _status[_is_single] = "single_lineage"
    _status[_is_doublet] = "doublet_suspected"
    _can_resolve = _is_multi & (_margin >= DOMINANCE_MARGIN) & (_td >= 2)
    _status[_is_multi & _can_resolve] = "lineage_phenotyped"
    _status[_is_multi & ~_can_resolve] = "lineage_ambiguous"

    _lin_arr = np.array(_lineages)
    _ct_orig = adata.obs["celltype"].astype(str).to_numpy()
    _ct_lineage = _ct_orig.copy()
    _resolved = _status == "lineage_phenotyped"
    _ct_lineage[_resolved] = _lin_arr[_top[_resolved]]
    _dominant = np.empty(adata.n_obs, dtype=object); _dominant[:] = ""
    _dominant[~_is_single] = _lin_arr[_top[~_is_single]]

    adata.obs["lineage_status"] = pd.Categorical(_status,
        categories=["single_lineage", "lineage_phenotyped",
                    "lineage_ambiguous", "doublet_suspected"])
    adata.obs["celltype_lineage"] = pd.Categorical(_ct_lineage)
    adata.obs["lineage_dominant"] = pd.Categorical(_dominant)
    adata.obs["lineage_top_depth"] = _td.astype(np.int8)
    adata.obs["lineage_second_depth"] = _sd.astype(np.int8)
    adata.obs["lineage_n_groups"] = _n_groups

    print("\n--- Lineage phenotyping ---")
    print(adata.obs["lineage_status"].value_counts().to_string())
    _diff = (_ct_orig != _ct_lineage).sum()
    print(f"  cells reclassified: {int(_diff):,}")
else:
    print("[skipped] lineage phenotyping — counts layer missing.")


## 10. Save Annotated Data

In [ ]:
output_file = OUTPUT_DIR / f"{SAMPLE_ID}_phenotyped.h5ad"
adata.write_h5ad(output_file)

print(f"\nPhenotyped data saved to: {output_file}")
print(f"Dataset shape: {adata.shape}")
print(f"\nCell type annotations:")
print(adata.obs['celltype'].value_counts())

celltype_export = adata.obs[['celltype', 'predicted_celltype', 'celltype_confidence']]
celltype_export.to_csv(OUTPUT_DIR / f"{SAMPLE_ID}_celltype_assignments.csv")
print(f"\nCell type assignments exported to: {OUTPUT_DIR / f'{SAMPLE_ID}_celltype_assignments.csv'}")

---

## Next Steps

The annotated data is ready for:
1. **Spatial analysis** (03_spatial_analysis.ipynb)
2. **Group-wise comparisons** (04_group_comparisons.ipynb)
3. **Tissue comparisons** (05_tissue_comparisons.ipynb)